<div style="text-align: center;">

  <img src="img/LogoUesc.png" alt="UESC" style="width: 60px; display: block; margin: 0 auto 5px auto;">

  <div style="font-size: 20px;"><strong>UNIVERSIDADE ESTADUAL DE SANTA CRUZ - UESC</strong></div>
  <div style="font-size: 18px;">DEPARTAMENTO DE ENGENHARIAS E COMPUTAÇÃO - DEC</div>
  <div style="font-size: 18px;">Grupo de Pesquisas em Simulações e Controle de Processos</div>

  <hr style="margin: 20px 0; border: 1px solid #0074B7;">

  <div style="font-size: 22px; font-weight: bold; margin-bottom: 10px;">PROGRAMAÇÃO EM PYTHON PARA AS ENGENHARIAS</div>

  <div style="font-size: 18px; margin-top: 10px;"><strong>Tema</strong>: Métodos Numéricos em Python — Solução Computacional de Problemas de Engenharia.</div>
  <div style="font-size: 18px;"><strong>Prof.  Dr.  E. R. Edwards</strong></div>

  <div style="font-size: 16px; margin-top: 15px;">Ilhéus - BA, Abril de 2027</div>
  
  <hr style="margin: 20px 0; border: 1px solid #0074B7;">

</div>

**Apresentação**

Os modelos de Engenharia frequentemente conduzem a equações que não podem ser resolvidas diretamente por manipulações algébricas simples. Os métodos numéricos permitem obter soluções aproximadas de forma sistemática, com controle de erro e verificação física.

Neste notebook, cada método será construído a partir de um problema de Engenharia: primeiro o fenômeno e a equação, depois um cálculo manual, a implementação do algoritmo em Python e, por último, quando pertinente, a comparação com funções de bibliotecas científicas. Os parâmetros são didáticos e não representam especificações industriais.

**Objetivos:** reconhecer erros numéricos, resolver equações não lineares e sistemas lineares, interpolar e ajustar medições, aproximar derivadas e integrais e integrar equações diferenciais simples.

**Sumário**

1. [Por que precisamos de métodos numéricos?](#secao-1)
2. [Preparação do ambiente e precisão numérica](#secao-2)
3. [Erros absoluto e relativo](#secao-3)
4. [Raízes de equações: método da bisseção](#secao-4)
5. [Raízes de equações: método de Newton-Raphson](#secao-5)
6. [Comparação de métodos e critérios de parada](#secao-6)
7. [Sistemas lineares: eliminação de Gauss](#secao-7)
8. [Sistemas lineares: solução com NumPy](#secao-8)
9. [Interpolação linear de dados experimentais](#secao-9)
10. [Ajuste de dados por mínimos quadrados](#secao-10)
11. [Diferenciação numérica de medições](#secao-11)
12. [Integração numérica: regra dos trapézios](#secao-12)
13. [Equações diferenciais: método de Euler](#secao-13)
14. [Estabilidade, passo e convergência](#secao-14)
15. [Aplicação integradora: tanque com escoamento](#secao-15)
16. [Exercícios de consolidação](#secao-16)
17. [Conclusões](#secao-17)

<div id="secao-1" style="background-color:#f1faff; border-left:6px solid #0074B7; color:#003366; border-radius:8px; padding:15px 20px; font-size:20px; font-weight:bold;">1. Por que precisamos de métodos numéricos?</div>

Um balanço de massa pode gerar uma equação não linear; um conjunto de balanços acoplados pode formar um sistema de equações; um instrumento pode fornecer dados apenas em instantes discretos. Em todos esses casos, precisamos de algoritmos para calcular uma aproximação e avaliar sua confiabilidade.

**Princípio fundamental:** um resultado com muitas casas decimais não é necessariamente preciso. É preciso verificar as hipóteses do modelo, a unidade física, o erro numérico e a sensibilidade aos dados.

<div id="secao-2" style="background-color:#f1faff; border-left:6px solid #0074B7; color:#003366; border-radius:8px; padding:15px 20px; font-size:20px; font-weight:bold;">2. Preparação do ambiente e precisão numérica</div>

Usaremos a biblioteca padrão `math` e, nos exemplos matriciais, o NumPy. Caso o NumPy não esteja instalado, utilize no terminal `python -m pip install numpy`.

Computadores armazenam números reais com precisão finita. Por isso, operações aparentemente exatas podem apresentar pequenas diferenças em ponto flutuante.

In [16]:
import math
import numpy as np

print("0.1 + 0.2 =", 0.1 + 0.2)
print("É exatamente igual a 0.3?", 0.1 + 0.2 == 0.3)
print("É numericamente próximo?", math.isclose(0.1 + 0.2, 0.3))

0.1 + 0.2 = 0.30000000000000004
É exatamente igual a 0.3? False
É numericamente próximo? True


**Interpretação:** a igualdade exata pode falhar devido à representação binária. Para comparar resultados de cálculos numéricos, é preferível definir tolerâncias adequadas ao problema.

<div id="secao-3" style="background-color:#f1faff; border-left:6px solid #0074B7; color:#003366; border-radius:8px; padding:15px 20px; font-size:20px; font-weight:bold;">3. Erros absoluto e relativo</div>

**Problema de Engenharia — Medição de temperatura.** Um padrão indica 80 °C e um sensor registra 79,6 °C. O erro absoluto é

$$
E_a=|T_{medido}-T_{referência}|=|79{,}6-80|=0{,}4 \ ^\circ \mathrm{C}.
$$

O erro relativo, quando a referência é diferente de zero, é

$$E_r=\ \frac{E_a}{|T_{referência}|}=\ \frac{0{,}4}{80}=0{,}005,$$

ou 0,5 %. Em metrologia, erro de medição e incerteza de medição não são conceitos equivalentes.

In [17]:
referencia = 80.0
medicao = 79.6
erro_absoluto = abs(medicao - referencia)
erro_relativo = erro_absoluto / abs(referencia)
print(f"Erro absoluto: {erro_absoluto:.2f}°C")
print(f"Erro relativo: {100 * erro_relativo:.2f}%")

Erro absoluto: 0.40°C
Erro relativo: 0.50%


<div id="secao-4" style="background-color:#f1faff; border-left:6px solid #0074B7; color:#003366; border-radius:8px; padding:15px 20px; font-size:20px; font-weight:bold;">4. Raízes de equações: método da bisseção</div>

**Problema de Engenharia — Determinação de uma condição operacional.** Considere a equação didática

$$f(x)=x^2-5=0,$$

na qual $x$ representa uma grandeza adimensional do modelo. Queremos determinar a raiz positiva. Sabemos que $f(2)=-1$ e $f(3)=4$; portanto, existe uma raiz entre 2 e 3, pois a função é contínua.

Na **bisseção**, calculamos o ponto médio $x_m=(a+b)/2$ e escolhemos o subintervalo em que a função muda de sinal. Na primeira iteração, $x_m=2{,}5$ e $f(2{,}5)=1{,}25$; logo, o novo intervalo é $[2;2{,}5]$.

In [18]:
def bissecao(f, a, b, tolerancia=1e-8, max_iter=100):
    if not (math.isfinite(a) and math.isfinite(b) and a < b):
        raise ValueError("Intervalo inválido.")
    fa, fb = f(a), f(b)
    if not (math.isfinite(fa) and math.isfinite(fb)):
        raise ValueError("Função não finita nos extremos.")
    if fa == 0:
        return a, 0
    if fb == 0:
        return b, 0
    if fa * fb > 0:
        raise ValueError("A função deve mudar de sinal no intervalo.")
    for k in range(1, max_iter + 1):
        meio = (a + b) / 2
        fm = f(meio)
        if not math.isfinite(fm):
            raise ValueError("Função não finita durante a iteração.")
        if abs(fm) <= tolerancia or (b - a) / 2 <= tolerancia:
            return meio, k
        if fa * fm < 0:
            b = meio
        else:
            a, fa = meio, fm
    raise RuntimeError("A bisseção não convergiu no limite de iterações.")

f = lambda x: x**2 - 5
raiz_b, passos_b = bissecao(f, 2.0, 3.0)
print(f"Raiz: {raiz_b:.9f}; iterações: {passos_b}")
print(f"Resíduo f(x): {f(raiz_b):.3e}")

Raiz: 2.236067973; iterações: 27
Resíduo f(x): -1.978e-08


**Interpretação:** a bisseção preserva um intervalo com mudança de sinal, o que torna o método robusto para funções contínuas. O resíduo `f(x)` e a largura do intervalo ajudam a avaliar a solução.

<div id="secao-5" style="background-color:#f1faff; border-left:6px solid #0074B7; color:#003366; border-radius:8px; padding:15px 20px; font-size:20px; font-weight:bold;">5. Raízes de equações: método de Newton-Raphson</div>

O método de **Newton-Raphson** utiliza a inclinação local da função:

$$x_{k+1}=x_k-\ \frac{f(x_k)}{f'(x_k)}.$$

Para $f(x)=x^2-5$, temos $f'(x)=2x$. Partindo de $x_0=2$, a primeira iteração é

$$x_1=2-\ \frac{2^2-5}{2(2)}=2{,}25.$$

A convergência costuma ser rápida perto de uma raiz simples, mas depende da estimativa inicial e exige cuidado quando a derivada se aproxima de zero.

In [19]:
def newton(f, derivada, x0, tolerancia=1e-10, max_iter=50):
    x = float(x0)
    for k in range(1, max_iter + 1):
        fx = f(x)
        dfx = derivada(x)
        if not (math.isfinite(fx) and math.isfinite(dfx)):
            raise ValueError("Valor não finito durante o método.")
        if abs(fx) <= tolerancia:
            return x, k - 1
        if abs(dfx) < 1e-14:
            raise ZeroDivisionError("Derivada muito próxima de zero.")
        novo = x - fx / dfx
        if not math.isfinite(novo):
            raise ValueError("Iteração não finita.")
        if abs(novo - x) <= tolerancia and abs(f(novo)) <= tolerancia:
            return novo, k
        x = novo
    raise RuntimeError("Newton não convergiu no limite de iterações.")

raiz_n, passos_n = newton(f, lambda x: 2*x, 2.0)
print(f"Raiz: {raiz_n:.10f}; iterações: {passos_n}")
print(f"Resíduo f(x): {f(raiz_n):.3e}")

Raiz: 2.2360679775; iterações: 4
Resíduo f(x): 8.882e-16


<div id="secao-6" style="background-color:#f1faff; border-left:6px solid #0074B7; color:#003366; border-radius:8px; padding:15px 20px; font-size:20px; font-weight:bold;">6. Comparação de métodos e critérios de parada</div>

Os dois métodos buscam a mesma raiz, mas apresentam comportamentos diferentes. A bisseção exige um intervalo com mudança de sinal; Newton exige uma derivada e uma boa estimativa inicial.

**Critérios de parada:** diferença entre iterações, valor absoluto do resíduo e limite de iterações. Nenhum critério isolado substitui a avaliação física do resultado. Quando a função tem unidades, a tolerância do resíduo também deve ser escolhida com unidades coerentes.

In [20]:
raiz_exata = math.sqrt(5)
for nome, raiz, passos in [("Bisseção", raiz_b, passos_b),
                           ("Newton", raiz_n, passos_n)]:
    print(f"{nome:9s}: erro = {abs(raiz-raiz_exata):.3e}; iterações = {passos}")

Bisseção : erro = 4.422e-09; iterações = 27
Newton   : erro = 0.000e+00; iterações = 4


<div id="secao-7" style="background-color:#f1faff; border-left:6px solid #0074B7; color:#003366; border-radius:8px; padding:15px 20px; font-size:20px; font-weight:bold;">7. Sistemas lineares: eliminação de Gauss</div>

**Problema de Engenharia — Balanços de massa acoplados.** Considere o sistema

$$2q_1+q_2=8, \ \qquad q_1+3q_2=9.$$

Na eliminação de Gauss, eliminamos $q_1$ da segunda equação. Subtraindo metade da primeira equação da segunda:

$$2{,}5q_2=5 \ \quad\ \Rightarrow\ \quad q_2=2.$$

Substituindo na primeira, $q_1=3$. A implementação a seguir realiza a eliminação com **pivotamento parcial**, seguida de substituição retroativa. O procedimento explicita o algoritmo antes do uso de uma biblioteca pronta.

In [21]:
def eliminacao_gauss(A, b):
    A = np.array(A, dtype=float, copy=True)
    b = np.array(b, dtype=float, copy=True)
    n = b.size
    if A.shape != (n, n):
        raise ValueError("A deve ser quadrada e compatível com b.")
    if not (np.isfinite(A).all() and np.isfinite(b).all()):
        raise ValueError("Dados não finitos.")
    escala = max(1.0, float(np.max(np.abs(A))))
    for k in range(n):
        pivo = k + int(np.argmax(np.abs(A[k:, k])))
        if abs(A[pivo, k]) <= 1e-12 * escala:
            raise ValueError("Sistema singular ou numericamente degenerado.")
        if pivo != k:
            A[[k, pivo]] = A[[pivo, k]]
            b[[k, pivo]] = b[[pivo, k]]
        for i in range(k+1, n):
            fator = A[i, k] / A[k, k]
            A[i, k:] -= fator * A[k, k:]
            b[i] -= fator * b[k]
    x = np.zeros(n)
    for i in range(n-1, -1, -1):
        x[i] = (b[i] - np.dot(A[i, i+1:], x[i+1:])) / A[i, i]
    return x

A = np.array([[2., 1.], [1., 3.]])
b = np.array([8., 9.])
q_gauss = eliminacao_gauss(A, b)
print("Vazões:", q_gauss)
print("Verificação A @ q:", A @ q_gauss)

Vazões: [3. 2.]
Verificação A @ q: [8. 9.]


**Interpretação:** o resultado é $q_1=3$ e $q_2=2$. A substituição na matriz original confirma os dois balanços. Em aplicações reais, também devemos verificar a viabilidade física das vazões.

<div id="secao-8" style="background-color:#f1faff; border-left:6px solid #0074B7; color:#003366; border-radius:8px; padding:15px 20px; font-size:20px; font-weight:bold;">8. Sistemas lineares: solução com NumPy</div>

Após compreender a eliminação, podemos usar `numpy.linalg.solve`, que implementa rotinas numéricas otimizadas. Essa função não deve ser confundida com a inversão explícita da matriz: para resolver $A\ \mathbf{x}=\ \mathbf{b}$, em geral é preferível resolver o sistema diretamente.

A **norma do resíduo** mede o quanto a solução reproduz as equações originais. O número de condição ajuda a identificar sensibilidade a pequenas perturbações nos dados.

In [22]:
q_numpy = np.linalg.solve(A, b)
residuo = A @ q_numpy - b
print("Solução NumPy:", q_numpy)
print("Norma do resíduo:", np.linalg.norm(residuo))
print("Número de condição:", np.linalg.cond(A))
print("Diferença entre algoritmos:", np.linalg.norm(q_numpy-q_gauss))

Solução NumPy: [3. 2.]
Norma do resíduo: 0.0
Número de condição: 2.6180339887498953
Diferença entre algoritmos: 0.0


<div id="secao-9" style="background-color:#f1faff; border-left:6px solid #0074B7; color:#003366; border-radius:8px; padding:15px 20px; font-size:20px; font-weight:bold;">9. Interpolação linear de dados experimentais</div>

**Problema de Engenharia — Calibração de instrumento.** Um instrumento registrou 20 °C em 4 mA e 100 °C em 20 mA. Admitindo linearidade entre esses pontos, qual temperatura corresponde a 12 mA?

A interpolação linear é

$$T(I)=T_1+\ \frac{T_2-T_1}{I_2-I_1}(I-I_1).$$

Para 12 mA,

$$T(12)=20+\ \frac{100-20}{20-4}(12-4)=60\\ ^\ \circ\ \mathrm{C}.$$

Interpolar entre medições é diferente de extrapolar para fora da faixa, onde o comportamento pode não ser linear.

In [23]:
correntes = np.array([4., 8., 12., 16., 20.])
temperaturas = np.interp(correntes, [4., 20.], [20., 100.])
for I, T in zip(correntes, temperaturas):
    print(f"{I:4.1f} mA -> {T:5.1f} °C")

 4.0 mA ->  20.0 °C
 8.0 mA ->  40.0 °C
12.0 mA ->  60.0 °C
16.0 mA ->  80.0 °C
20.0 mA -> 100.0 °C


<div id="secao-10" style="background-color:#f1faff; border-left:6px solid #0074B7; color:#003366; border-radius:8px; padding:15px 20px; font-size:20px; font-weight:bold;">10. Ajuste de dados por mínimos quadrados</div>

**Problema de Engenharia — Ajuste de dados experimentais.** Considere três medições de um sistema térmico: $(t,T)=(0,20),(1,23),(2,27)$, com tempo em minutos e temperatura em °C. Procuramos uma reta

$$
T(t)=a+bt
$$

que minimize a soma dos quadrados dos resíduos,

$$
S(a,b)=\ \sum_i[T_i-(a+bt_i)]^2.
$$

Pelas equações normais, $\ \sum T_i=na+b\ \sum t_i$ e $\ \sum t_iT_i=a\ \sum t_i+b\ \sum t_i^2$. Como $\ \sum t_i=3$, $\ \sum t_i^2=5$, $\ \sum T_i=70$ e $\ \sum t_iT_i=77$, temos $3a+3b=70$ e $3a+5b=77$. Subtraindo, $2b=7$, logo $b=3{,}5$ °C/min e $a\ \approx19{,}833$ °C.

Ajuste estatístico não demonstra, por si só, uma relação causal.

In [24]:
tempos = np.array([0., 1., 2.])
T_medidas = np.array([20., 23., 27.])

X = np.column_stack([np.ones_like(tempos), tempos])
coeficientes, *_ = np.linalg.lstsq(X, T_medidas, rcond=None)
a, b_coef = coeficientes
T_ajustadas = X @ coeficientes
print(f"T(t) = {a:.4f} + {b_coef:.4f} t")
print("Resíduos (°C):", T_medidas - T_ajustadas)

T(t) = 19.8333 + 3.5000 t
Resíduos (°C): [ 0.16666667 -0.33333333  0.16666667]


<div id="secao-11" style="background-color:#f1faff; border-left:6px solid #0074B7; color:#003366; border-radius:8px; padding:15px 20px; font-size:20px; font-weight:bold;">11. Diferenciação numérica de medições</div>

**Problema de Engenharia — Taxa de aquecimento.** A derivada $dT/dt$ representa a taxa instantânea de variação da temperatura. Quando dispomos apenas de dados discretos, podemos aproximá-la por diferenças finitas.

Na diferença progressiva,

$$\ \left.\ \frac{dT}{dt}\ \right|_{t_i}\ \approx\ \frac{T_{i+1}-T_i}{\ \Delta t}.$$

Se a temperatura aumenta de 20 para 23 °C em 1 min, a taxa média no intervalo é 3 °C/min. Para pontos internos igualmente espaçados, a diferença central costuma ter erro de truncamento menor:

$$\ \left.\ \frac{dT}{dt}\ \right|_{t_i}\ \approx\ \frac{T_{i+1}-T_{i-1}}{2\ \Delta t}.$$

Derivar medições ruidosas pode amplificar o ruído; não devemos interpretar cada oscilação numérica como fenômeno físico.

In [25]:
t = np.array([0., 1., 2., 3.])
T = np.array([20., 23., 27., 32.])
taxas_intervalos = np.diff(T) / np.diff(t)
taxa_central_t1 = (T[2]-T[0])/(t[2]-t[0])
print("Taxas médias por intervalo (°C/min):", taxas_intervalos)
print(f"Diferença central em t=1 min: {taxa_central_t1:.2f} °C/min")

Taxas médias por intervalo (°C/min): [3. 4. 5.]
Diferença central em t=1 min: 3.50 °C/min


<div id="secao-12" style="background-color:#f1faff; border-left:6px solid #0074B7; color:#003366; border-radius:8px; padding:15px 20px; font-size:20px; font-weight:bold;">12. Integração numérica: regra dos trapézios</div>

**Problema de Engenharia — Energia transferida ao longo do tempo.** Um equipamento fornece potências térmicas de 100, 120 e 140 W nos instantes 0, 10 e 20 s. A energia transferida é

$$
E= \int_0^{20}P(t)\,dt.
$$

Pela regra dos trapézios, para intervalos de 10 s,

$$
E \approx \frac{10}{2}(100+120)+ \frac{10}{2}(120+140)=2400\ \mathrm{J}
$$

A integração numérica aproxima a área sob a curva. O resultado depende da resolução temporal e da forma do sinal entre medições.

In [26]:
tempo_s = np.array([0., 10., 20.])
potencia_W = np.array([100., 120., 140.])
energia_J = np.sum((potencia_W[:-1]+potencia_W[1:])/2*np.diff(tempo_s))
print(f"Energia por trapézios: {energia_J:.0f} J")

Energia por trapézios: 2400 J


<div id="secao-13" style="background-color:#f1faff; border-left:6px solid #0074B7; color:#003366; border-radius:8px; padding:15px 20px; font-size:20px; font-weight:bold;">13. Equações diferenciais: método de Euler</div>

**Problema de Engenharia — Resfriamento de um corpo.** Admitindo temperatura uniforme no corpo, propriedades constantes e ambiente a temperatura constante, o modelo concentrado é

$$\ \frac{dT}{dt}=-k(T-T_a),$$

com $k$ em min$^{-1}$. Pelo método de Euler explícito,

$$T_{n+1}=T_n+\ \Delta t[-k(T_n-T_a)].$$

Considere $T_0=80 °C$,  $ \ T_a=20 °C$, $ \ k=0{,}1 \ min^{-1}$ e $ \Delta t=1 \ min$. O primeiro passo manual fornece

$$
T_1=80-0{,}1(80-20)=74 \ ^\circ \mathrm{C}.
$$

Vamos implementar o algoritmo sem recorrer a um integrador pronto.

In [27]:
T0 = 80.0
Ta = 20.0
k_resfriamento = 0.1
dt = 1.0
n_passos = 10
tempo = np.arange(n_passos+1)*dt
T_euler = np.empty(n_passos+1)
T_euler[0] = T0
for n in range(n_passos):
    T_euler[n+1] = T_euler[n] - dt*k_resfriamento*(T_euler[n]-Ta)

T_analitica = Ta + (T0-Ta)*np.exp(-k_resfriamento*tempo)
print("Primeiro passo Euler:", T_euler[1], "°C")
print(f"Em 10 min: Euler={T_euler[-1]:.3f} °C; exata={T_analitica[-1]:.3f} °C")

Primeiro passo Euler: 74.0 °C
Em 10 min: Euler=40.921 °C; exata=42.073 °C


**Interpretação:** Euler produz uma aproximação para a trajetória térmica. A solução analítica permite quantificar o erro neste exemplo, mas nem todos os modelos de Engenharia possuem solução fechada.

<div id="secao-14" style="background-color:#f1faff; border-left:6px solid #0074B7; color:#003366; border-radius:8px; padding:15px 20px; font-size:20px; font-weight:bold;">14. Estabilidade, passo e convergência</div>

**Problema de Engenharia — Influência do passo.** O passo temporal afeta a precisão e pode afetar a estabilidade. Para o modelo $dT/dt=-k(T-T_a)$, Euler explícito satisfaz

$$T_{n+1}-T_a=(1-k\ \Delta t)(T_n-T_a).$$

A condição de estabilidade linear é $|1-k\ \Delta t|<1$, isto é, $0<k\ \Delta t<2$. Para evitar oscilações numéricas neste modelo, uma condição mais restritiva é $0<k\ \Delta t\ \leq1$.

Diminuir o passo tende a reduzir o erro de discretização, mas exige mais operações. Vamos comparar diferentes passos no mesmo instante final.

In [28]:
def euler_resfriamento(dt, tf=10.0, T0=80.0, Ta=20.0, k=0.1):
    n = round(tf/dt)
    if n <= 0 or not math.isclose(n*dt, tf, rel_tol=1e-12, abs_tol=1e-12):
        raise ValueError("O passo deve dividir o tempo final.")
    T = T0
    for _ in range(n):
        T += dt*(-k*(T-Ta))
    return T

exata_10 = 20+60*math.exp(-1)
for passo in [2.0, 1.0, 0.5, 0.25]:
    aproximacao = euler_resfriamento(passo)
    print(f"dt={passo:4.2f} min; T(10)={aproximacao:.5f} °C; erro={abs(aproximacao-exata_10):.5f} °C")

dt=2.00 min; T(10)=39.66080 °C; erro=2.41197 °C
dt=1.00 min; T(10)=40.92071 °C; erro=1.15206 °C
dt=0.50 min; T(10)=41.50916 °C; erro=0.56361 °C
dt=0.25 min; T(10)=41.79395 °C; erro=0.27882 °C


<div id="secao-15" style="background-color:#f1faff; border-left:6px solid #0074B7; color:#003366; border-radius:8px; padding:15px 20px; font-size:20px; font-weight:bold;">15. Aplicação integradora: tanque com escoamento</div>

**Problema integrador — Tanque com entrada e saída.** Um tanque recebe $Q_{in}=5$ L/min e descarrega $Q_{out}=4$ L/min. Seu volume inicial é $V_0=50$ L. O balanço global, sem reação e com vazões constantes, é

$$\ \frac{dV}{dt}=Q_{in}-Q_{out}.$$

A solução analítica é

$$V(t)=V_0+(Q_{in}-Q_{out})t.$$

Em 10 min, $V(10)=50+(5-4)10=60$ L. Aplicando Euler com passo de 1 min,

$$V_{n+1}=V_n+\ \Delta t(Q_{in}-Q_{out}).$$

Como a derivada é constante, Euler reproduz exatamente a solução matemática, salvo arredondamentos. O modelo só é fisicamente válido enquanto não houver transbordamento, esvaziamento ou mudança das vazões.

In [29]:
V0 = 50.0
Qin, Qout = 5.0, 4.0
dt = 1.0
tempo_min = np.arange(0., 11., dt)
V = np.empty(tempo_min.size)
V[0] = V0
for n in range(len(V)-1):
    V[n+1] = V[n] + dt*(Qin-Qout)
V_exato = V0 + (Qin-Qout)*tempo_min
print("Volumes (L):", V)
print("Volume final:", V[-1], "L")
print("Erro máximo:", np.max(np.abs(V-V_exato)), "L")

Volumes (L): [50. 51. 52. 53. 54. 55. 56. 57. 58. 59. 60.]
Volume final: 60.0 L
Erro máximo: 0.0 L


**Interpretação:** o tanque acumula 1 L/min, alcançando 60 L em 10 min. Antes de utilizar o modelo em uma instalação real, seria necessário conhecer a capacidade do tanque, limites operacionais e comportamento das vazões.

<div id="secao-16" style="background-color:#f1faff; border-left:6px solid #0074B7; color:#003366; border-radius:8px; padding:15px 20px; font-size:20px; font-weight:bold;">16. Exercícios de consolidação</div>

**Exercício 1 — Erros numéricos.** Um instrumento registra 48,7 °C quando a referência é 50 °C. Calcule manualmente os erros absoluto e relativo e implemente a verificação em Python.

**Exercício 2 — Bisseção.** Determine a raiz positiva de $x^2-7=0$ no intervalo $[2,3]$. Mostre as três primeiras iterações manualmente e compare o resultado com `math.sqrt(7)`.

**Exercício 3 — Newton-Raphson.** Resolva $x^3-2x-5=0$ a partir de $x_0=2$. Escreva a derivada e explique o significado do resíduo.

**Exercício 4 — Balanços acoplados.** Resolva por eliminação de Gauss o sistema $3q_1+q_2=11$ e $q_1+2q_2=7$. Confirme com NumPy.

**Exercício 5 — Instrumentação.** Interpole linearmente uma leitura de pressão para 12 mA, considerando 4 mA = 0 bar e 20 mA = 8 bar. Explique a limitação da extrapolação.

**Exercício 6 — Ajuste experimental.** Ajuste uma reta aos dados $(0,30)$, $(1,34)$ e $(2,39)$, com tempo em minutos e temperatura em °C. Interprete o coeficiente angular.

**Exercício 7 — Transferência de Calor.** Integre numericamente uma potência medida em 0, 5, 10 e 15 s, com valores 80, 90, 110 e 120 W, usando trapézios.

**Exercício 8 — Dinâmica de processos.** Resolva $dT/dt=-0{,}2(T-25)$ com $T(0)=75$ °C pelo método de Euler até 5 min, usando passos de 1 e 0,5 min. Compare com a solução analítica.

**Desafio — Engenharia de Processos.** Modifique o modelo do tanque para uma vazão de saída variável $Q_{out}=0{,}05V$ (em L/min quando $V$ está em L). Escreva a equação diferencial, implemente Euler, justifique o passo e discuta os limites físicos.

<div id="secao-17" style="background-color:#f1faff; border-left:6px solid #0074B7; color:#003366; border-radius:8px; padding:15px 20px; font-size:20px; font-weight:bold;">17. Conclusões</div>

Neste notebook, desenvolvemos algoritmos para encontrar raízes, resolver sistemas lineares, interpolar e ajustar dados, aproximar derivadas e integrais e integrar equações diferenciais. O objetivo não é memorizar funções, mas **entender como a aproximação é construída, quando ela é válida e como verificar o resultado**.

O próximo notebook aprofundará a modelagem e a simulação de sistemas dinâmicos, partindo das equações diferenciais que descrevem fenômenos de Engenharia.

<div style="text-align: center; font-size: 12px; color: gray; margin-top: 40px; padding-top: 15px; border-top: 1px solid #dddddd;">
Material desenvolvido pelo Grupo de Pesquisas em Modelagem Computacional da UESC.<br>
© 2026 — Todos os direitos reservados. Uso comercial somente mediante autorização.
</div>